# 02f Donor-Bound Tokenizer Generation

## Purpose

This notebook builds the donor-bound glycan tokenizer for the rebuilt workflow.

## Input

- `MyDrive/GlycanProject/data/splits/train.txt`

## Outputs

- `MyDrive/GlycanProject/tokenizers/donor_bound/<setting_label>/vocab.json`
- Hugging Face tokenizer files saved in the same folder
- `tokenizer_config_summary.json`
- `inspection_preview.csv`

## Notes to myself

This tokenizer keeps the donor sugar, anomer, and donor carbon together as one unit such as `Galb1` or `Neu5Aca2`, while splitting the acceptor carbon into a separate token such as `-4` or `-3`.


## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- tokenizer artifacts stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub


In [ ]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

# Mount Google Drive so the notebook can read data files and save outputs.
drive.mount('/content/drive')

# Clone the public GitHub repository into the Colab runtime.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}
!git pull origin main --no-edit -q

# Add the repo to the Python path so src/ imports work across notebooks.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')


## Path and setting setup

This tokenizer has a fixed vocabulary rather than a merge schedule, so I'm using the same `v1_train_only` label pattern as the other deterministic tokenizers. The donor-bound vocabulary is built directly from the training split only.


In [ ]:
# ==============================================================================
# 1. DEFINE THE TRAINING PATHS AND TOKENIZER SETTINGS
# ==============================================================================
PROJECT_ROOT = '/content/drive/MyDrive/GlycanProject'
TRAIN_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'splits', 'train.txt')

SETTING_LABEL = 'v1_train_only'
TOKENIZER_OUT_DIR = os.path.join(PROJECT_ROOT, 'tokenizers', 'donor_bound', SETTING_LABEL)
VOCAB_PATH = os.path.join(TOKENIZER_OUT_DIR, 'vocab.json')

os.makedirs(TOKENIZER_OUT_DIR, exist_ok=True)

print('Training data path:')
print(TRAIN_DATA_PATH)
print('\nTokenizer output directory:')
print(TOKENIZER_OUT_DIR)
print('\nSetting label:')
print(SETTING_LABEL)

if not os.path.exists(TRAIN_DATA_PATH):
    raise FileNotFoundError(f'Training split not found: {TRAIN_DATA_PATH}')


## Build the donor-bound vocabulary

This step applies donor-bound tokenization to every training glycan, counts the token inventory, and writes a stable `vocab.json`. Units such as `Galb1` and `Neu5Aca2` should stay together, while acceptor carbons like `-4` and `-3` stay separate.


In [ ]:
# ==============================================================================
# 2. BUILD AND SAVE THE DONOR-BOUND VOCABULARY
# ==============================================================================
from src.tokenizer_utils import (
    SPECIAL_TOKENS,
    build_wordlevel_vocab_from_sequences,
    load_sequences_from_text,
    save_vocab_json,
    split_glycan_string_donor_bound,
)

train_sequences = load_sequences_from_text(TRAIN_DATA_PATH)
vocab, token_counts = build_wordlevel_vocab_from_sequences(
    train_sequences,
    split_glycan_string_donor_bound,
)

save_vocab_json(vocab, VOCAB_PATH)

print(f'Donor-bound vocabulary saved to: {VOCAB_PATH}')
print(f'Vocabulary size: {len(vocab)}')
print(f'Unique non-special tokens: {len(vocab) - len(SPECIAL_TOKENS)}')


## Compile the Hugging Face tokenizer

Here I'm wrapping the donor-bound vocabulary in a Hugging Face fast tokenizer. The regex pre-tokenizer is built directly from the learned vocabulary so the saved tokenizer follows the same donor-plus-acceptor split as the vocab-building step.


In [ ]:
# ==============================================================================
# 3. BUILD AND SAVE THE DONOR-BOUND HUGGING FACE TOKENIZER
# ==============================================================================
import json

from src.tokenizer_utils import (
    build_vocab_regex_from_wordlevel_vocab,
    create_wordlevel_fast_tokenizer,
)

with open(VOCAB_PATH, 'r', encoding='utf-8') as file:
    vocab = json.load(file)

PRETOKENIZER_PATTERN = build_vocab_regex_from_wordlevel_vocab(vocab)
backend_tokenizer, hf_tokenizer = create_wordlevel_fast_tokenizer(
    vocab,
    PRETOKENIZER_PATTERN,
)

hf_tokenizer.save_pretrained(TOKENIZER_OUT_DIR)

tokenizer_summary = {
    'tokenizer_family': 'donor_bound',
    'setting_label': SETTING_LABEL,
    'train_data_path': TRAIN_DATA_PATH,
    'tokenizer_output_dir': TOKENIZER_OUT_DIR,
    'pretokenizer_pattern': PRETOKENIZER_PATTERN,
    'vocab_size': len(vocab),
    'num_special_tokens': len(SPECIAL_TOKENS),
    'num_non_special_tokens': len(vocab) - len(SPECIAL_TOKENS),
    'top_10_tokens': token_counts.most_common(10),
    'saved_files': sorted(os.listdir(TOKENIZER_OUT_DIR)),
}

summary_json_path = os.path.join(TOKENIZER_OUT_DIR, 'tokenizer_config_summary.json')
with open(summary_json_path, 'w', encoding='utf-8') as file:
    json.dump(tokenizer_summary, file, indent=2)

print('Donor-bound tokenizer saved.')
print(f'Tokenizer folder: {TOKENIZER_OUT_DIR}')


## Quick sanity check

I don't want deep analysis here. I just want to confirm the saved tokenizer loads and that the first few glycans are being split into donor-bound units plus separate acceptor tokens.


In [ ]:
# ==============================================================================
# 4. LOAD THE SAVED TOKENIZER AND INSPECT SAMPLE OUTPUT
# ==============================================================================
import pandas as pd
from transformers import PreTrainedTokenizerFast

loaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_OUT_DIR)

sample_sequences = train_sequences[:3]
inspection_rows = []

for sample_index, sequence in enumerate(sample_sequences, start=1):
    token_ids = loaded_tokenizer.encode(sequence, add_special_tokens=False)
    tokens = loaded_tokenizer.convert_ids_to_tokens(token_ids)

    inspection_rows.append(
        {
            'sample_index': sample_index,
            'sequence': sequence,
            'num_tokens': len(tokens),
            'tokens': ' | '.join(tokens[:30]),
        }
    )

inspection_df = pd.DataFrame(inspection_rows)
display(inspection_df)

print(f'Loaded vocabulary size: {len(loaded_tokenizer)}')
print(f'Mask token: {loaded_tokenizer.mask_token}')
print(f'Pad token: {loaded_tokenizer.pad_token}')


## Save a small inspection table

This gives me a lightweight record of the first donor-bound sanity check without reopening the notebook later.


In [ ]:
# ==============================================================================
# 5. SAVE THE INSPECTION OUTPUT
# ==============================================================================
inspection_path = os.path.join(TOKENIZER_OUT_DIR, 'inspection_preview.csv')
inspection_df.to_csv(inspection_path, index=False)

print(f'Inspection preview saved to: {inspection_path}')
